# Load dependencies

In [ ]:
import matplotlib.pyplot as plt, torch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from torch.distributions.multivariate_normal import MultivariateNormal
from torch import tensor


torch.manual_seed(42)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)

# Create data

In [ ]:
n_clusters=6
n_samples =250
synthetic_data_mean = 30
synthetic_data_std = 50
center_cov_mat = torch.diag(tensor([5.,5.]))

centroids = (torch.rand(n_clusters, 2) - 0.5) * synthetic_data_std + synthetic_data_mean

def sample(m):
  return MultivariateNormal(m, center_cov_mat).sample((n_samples,))

slices = [sample(c) for c in centroids]
data = torch.cat(slices)

# Inefficient implementation

In [ ]:
X = data.clone()
n_cluster = 6
# Select k random elements along axis 0
indices = torch.randint(high=X.shape[0], size=(n_cluster,))
current_cents = X[indices]
print(f"current_cents: {current_cents}")

n_cluster=current_cents.shape[0]
# update centroid assignment
cent_map_pt = torch.zeros(X.shape[0])
for i, x in enumerate(X):
    dist = ((x-current_cents)**2).sum(-1)#.sqrt() # Broadcast
    cent_map_pt[i] = dist.argmin(0)

# update centroid coordinates
new_cents = torch.stack([
    X[cent_map_pt == i].mean(axis=0)
    for i in range(n_cluster)
])
print(f"new centroids: {new_cents}")

In [ ]:
def one_update(X, current_cents):
    n_cluster=current_cents.shape[0]
    # update centroid assignment
    cent_map_pt = torch.zeros(X.shape[0])
    for i, x in enumerate(X):
        dist = ((x-current_cents)**2).sum(-1)#.sqrt() # Broadcast
        cent_map_pt[i] = dist.argmin(0)

    # update centroid coordinates
    new_cent_list = []
    for i in range(n_cluster):
        new_cent = X[cent_map_pt == i].mean(axis=0)
        if new_cent.isnan().any(): new_cent = current_cents[i]
        new_cent_list.append(new_cent)
    new_cents = torch.stack(new_cent_list)
    return new_cents

print(one_update(X, current_cents))

In [ ]:
def _kmeanspp_init(X, n_cluster, threshold=0.9):
    n = X.shape[0]
    device = X.device
    # 1) first centroid uniformly at random
    first_idx = torch.randint(0, n, (1,), device=device).squeeze(0)
    centroids = [X[first_idx]]

    chosen = torch.zeros(n, dtype=torch.bool, device=device)
    chosen[first_idx] = True

    # 2) next centroids: sample proportional to distance^2 to nearest centroid
    for _ in range(1, n_cluster):
        C = torch.stack(centroids)  # [k, d]
        d2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)  # [n, k]
        min_d2 = d2.min(dim=1).values  # [n]

        min_d2[min_d2 < min_d2.quantile(threshold)] = 0.0

        # avoid re-picking same index
        min_d2[chosen] = 0.0
        total = min_d2.sum()

        if total <= 0:
            # fallback: pick uniformly from unchosen points
            candidates = (~chosen).nonzero(as_tuple=False).squeeze(1)
            idx = candidates[torch.randint(0, candidates.numel(), (1,), device=device)]
        else:
            probs = min_d2 / total
            idx = torch.multinomial(probs, num_samples=1)

        idx = idx.squeeze(0)
        centroids.append(X[idx])
        chosen[idx] = True

    return torch.stack(centroids)

In [ ]:
def kmeans(data, n_cluster=6, n_iter=5, return_history=False):
    X = data.clone()
    # indices = torch.randint(high=X.shape[0], size=(n_cluster,))
    # current_cents = X[indices]
    current_cents = _kmeanspp_init(X, n_cluster) # stable init (kmeans++)

    history = [current_cents.clone()] if return_history else None
    for _ in range(n_iter):
        current_cents = one_update(X, current_cents)
        if return_history:
            history.append(current_cents.clone())

    if return_history:
        history = torch.stack(history) # [T, K, 2]
        return (current_cents, history)
    else:
        return current_cents

# %timeit -n 10 centroids=kmeans(data)

# Plot data

In [ ]:
def plot_data(centroids, data, n_samples, ax=None):
    if ax is None: _,ax = plt.subplots()
    for i, centroid in enumerate(centroids):
        samples = data[i*n_samples:(i+1)*n_samples]
        ax.scatter(samples[:,0], samples[:,1], s=1)
        ax.plot(*centroid, markersize=10, marker="x", color='k', mew=5)
        ax.plot(*centroid, markersize=5, marker="x", color='m', mew=2)

In [ ]:
centroids=kmeans(data)
plot_data(centroids, data, n_samples)

# Animation

In [ ]:
def animate_kmeans(X, cent_history, interval=600):
    X_np = X.clone().cpu().numpy()
    H = cent_history.clone().cpu().numpy()
    K = H.shape[1]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(X_np[:, 0], X_np[:, 1], s=20, c="lightgray", alpha=0.8)
    cents_sc = ax.scatter([], [], s=180, c="red", marker="X", edgecolor="black")

    # trajectory line: from t-1 -> t
    trail_lines = [ax.plot([], [], '-', lw=2, alpha=0.8)[0] for _ in range(K)]

    ax.set_xlim(X_np[:, 0].min() - 1, X_np[:, 0].max() + 1)
    ax.set_ylim(X_np[:, 1].min() - 1, X_np[:, 1].max() + 1)

    def update(t):
        C = H[t]
        cents_sc.set_offsets(C)

        for j in range(K):
            if t == 0:
                trail_lines[j].set_data([], [])
            else:
                seg = H[:t+1, j, :]
                trail_lines[j].set_data(seg[:, 0], seg[:, 1])

        ax.set_title(f"KMeans centroid evolution - iter {t}")
        return (cents_sc, *trail_lines)

    ani = FuncAnimation(fig, update, frames=H.shape[0], interval=interval, blit=True, repeat=False)
    plt.close(fig)
    return ani

In [ ]:
n_paths = 1
final_cents_list = []
hist_list = []
for _ in range(n_paths):
    final_cents, hist = kmeans(data, return_history=True)
    final_cents_list.append(final_cents)
    hist_list.append(hist)

hist = torch.stack(hist_list).mean(0)
final_cents = torch.stack(final_cents_list).mean(0)
ani = animate_kmeans(data, hist, interval=400)
HTML(ani.to_jshtml())

# GPU batched algorithm

To truly accelerate the algorithm, we need to be performing updates on a batch of points per iteration, instead of just one as we were doing.

In [ ]:
def one_update_b(X, current_cents, bs):
    n = len(X)
    n_cluster=current_cents.shape[0]
    # update centroid assignment
    cent_map_pt = torch.zeros(X.shape[0])
    for i in range(0, n, bs):
        s = slice(i, min(i+bs,n))
        d2 = ((X[s][:,None]-current_cents[None])**2).sum(-1)
        cent_map_pt[s] = d2.argmin(1)
    # update centroid coordinates
    new_cent_list = []
    for i in range(n_cluster):
        new_cent = X[cent_map_pt == i].mean(axis=0)
        if new_cent.isnan().any(): new_cent = current_cents[i]
        new_cent_list.append(new_cent)
    new_cents = torch.stack(new_cent_list)
    return new_cents

In [ ]:
def kmeans_b(
    data,
    bs=300,
    n_cluster=6,
    n_iter=5,
    return_history=False
):
    X = data.clone()
    current_cents = _kmeanspp_init(X, n_cluster)
    history = [current_cents.clone()] if return_history else None
    for _ in range(n_iter):
        current_cents = one_update_b(X, current_cents, bs)
        if return_history:
            history.append(current_cents.clone())

    if return_history:
        history = torch.stack(history) # [T, K, 2]
        return (current_cents, history)
    else:
        return current_cents

## Animate results

In [ ]:
n_paths = 1
final_cents_list = []
hist_list = []
for _ in range(n_paths):
    final_cents, hist = kmeans_b(data, return_history=True)
    final_cents_list.append(final_cents)
    hist_list.append(hist)

hist = torch.stack(hist_list).mean(0)
final_cents = torch.stack(final_cents_list).mean(0)
ani = animate_kmeans(data, hist, interval=400)
HTML(ani.to_jshtml())

## Compare runtime (CPU)

In [ ]:
%timeit -n 10 final_cents = kmeans(data)

In [ ]:
%timeit -n 10 final_cents = kmeans_b(data, bs=500)

## Compare runtime (GPU)

In [ ]:
data = data.cuda()
%timeit -n 10 final_cents = kmeans_b(data, bs=500).cpu()

In [ ]:
# TODO: try to make the current implementation more efficient
    # remove loops and vectorize as much as possible.
    # try to make it benefit more from GPU
# TODO: try to implement the same logic in APL